<a href="https://colab.research.google.com/github/mounikas-tech-AI/Deeplearning_Projects/blob/DL/OCR/pilot_easyOCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Information Extraction Pipeline

This notebook automates the extraction of structured pilot details from unstructured NTSB aviation accident PDF reports using Python, PDFPlumber, EasyOCR, and Multiprocessing.

## Project Overview

When extracting data from official PDF reports, two main challenges arise:
1. **Scanned Images (Non-selectable Text):** Some PDFs contain images rather than selectable text layouts.
2. **Varying Multi-column Structures:** Reports may contain metadata for pilots, co-pilots, or multiple passengers. This pipeline targets the primary **Pilot Information** section and isolates it to avoid overlapping data fields.

### 🚀 Features Included
- **OCR Fallback:** Falls back automatically to `EasyOCR` when standard PDF text extraction yields empty text.
- **Optimized NLP Regex Parsing:** Robust regular expressions designed to capture multi-line structural details (such as *Flight Time* and *Last Flight Review*).
- **Multiprocessing Execution:** Speeds up file reading by mapping tasks across multiple CPU cores using `ProcessPoolExecutor`.
- **Data Standardization:** Outputs clean structured data into a pandas DataFrame and saves it directly to Google Drive as a CSV file.

---

## 🛠️ Step 1: Environment Setup & Library Installation

In [ ]:
pip install easyocr


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.9/422.9 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

## 📁 Step 2: Google Drive Mounting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 34.1 MB/s eta 0:00:00


## ⚙️ Step 3: Core Extraction Engine & Regex Pipeline

Here we establish the core extraction routines, initialize the OCR engine, use parallel processing to read the files, and structure the data.

In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            if "Certificate:" in page_text:
                text.append(page_text)
    return "\n".join(text)

# Function to extract pilot info (same as before)
def extract_pilot_info(text, file_name):
    pattern = r"""
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Last\sFlight\sReview\sor\sEquivalent:\s*(.*?)\s*
        Flight\sTime:\s*((?:.*?\n?)+?)\s*(?:\n[A-Z]|$)  # Capture multi-line Flight Time
    """

    match = re.search(pattern, text, re.S | re.VERBOSE)
    if match:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
                "Last Flight Review or Equivalent", "Flight Time"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]

        values[-1] = re.sub(r"\s*\n\s*", " ", values[-1])  # Ensure Flight Time is cleaned properly

        pilot_info = dict(zip(keys, values))
        pilot_info["File Name"] = file_name  # Add file name for tracking
        return pilot_info

    return None


# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.append(res)

# Convert to DataFrame
pilot_df = pd.DataFrame(pilot_data)

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot_Information.csv"
pilot_df.to_csv(output_csv, index=False)
print(f"Extracted pilot information saved to {output_csv}")


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteExtracted pilot information saved to /content/drive/MyDrive/Pilot_Information.csv


In [ ]:
pilot_df

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Last Flight Review or Equivalent,Flight Time,File Name
0,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No,"October 20, 2022","(Estimated) 639.49 hours (Total, all aircraft)...",Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
1,Private,"54,Male",Single-engine land,Right,None,Lap only,None,Yes,None,Yes,Class 2 With waivers/limitations,"January 19, 2022",No,,"(Estimated) 1310 hours (Total, all aircraft) P...",Report_CEN22FA298_105410_2_3_2025 7_50_00 AM.pdf
2,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No,,Page 5 of 8 CEN23LA211 Aircraft and Owner/Oper...,Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
3,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes,,Page 4 of 7 CHI01FA329 Aircraft and Owner/Oper...,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
4,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No,,(Estimated) Pilot Information (A2) Certificate...,Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
5,Commercial,"45,Male",Single-engine land; Single-engine,Front sea; Multi-engine land,Helicopter,3-point,Airplane,No,None,,Class 2 With waivers/limitations,"November 16, 2022",Yes,"April 19, 2023","10570 hours (Total, all aircraft), 398 hours (...",Report_ANC23LA032_139291_2_3_2025 7_42_32 AM.pdf


In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            if "Certificate:" in page_text:
                text.append(page_text)
    return "\n".join(text)

# Function to extract pilot info
def extract_pilot_info(text, file_name):
    pattern = r"""
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Last\sFlight\sReview\sor\sEquivalent:\s*(.*?)\s*
        Flight\sTime:\s*(.*?)\s*
    """

    match = re.search(pattern, text, re.S | re.VERBOSE)
    if match:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
                "Last Flight Review or Equivalent"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]

        pilot_info = dict(zip(keys, values))

        # Extract Flight Time separately
        flight_time_match = re.search(r"Flight Time:\s*(.*)", text, re.S | re.MULTILINE)
        pilot_info["Flight Time"] = re.sub(r"\s*\n\s*", " ", flight_time_match.group(1).strip()) if flight_time_match else "Not Found"

        # Move "Last Flight Review or Equivalent" after "Flight Time"
        last_flight_review = pilot_info.pop("Last Flight Review or Equivalent", "Not Found")
        pilot_info["Flight Time"] = pilot_info.pop("Flight Time", "Not Found")
        pilot_info["Last Flight Review or Equivalent"] = last_flight_review

        # Add File Name **after Last Flight Review or Equivalent**
        pilot_info["File Name"] = file_name

        return pilot_info

    return None

# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.append(res)

# Convert to DataFrame
pilot_df = pd.DataFrame(pilot_data)

# Ensure column order: Flight Time → Last Flight Review or Equivalent → File Name
column_order = [col for col in pilot_df.columns if col not in ["Last Flight Review or Equivalent","Flight Time", "File Name"]] \
               + ["Last Flight Review or Equivalent","Flight Time", "File Name"]
pilot_df = pilot_df[column_order]

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot3_Information.csv"
pilot_df.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot3_Information.csv


In [ ]:
pilot_df

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Last Flight Review or Equivalent,Flight Time,File Name
0,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No,"October 20, 2022","(Estimated) 639.49 hours (Total, all aircraft)...",Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
1,Private,"54,Male",Single-engine land,Right,None,Lap only,None,Yes,None,Yes,Class 2 With waivers/limitations,"January 19, 2022",No,,"(Estimated) 1310 hours (Total, all aircraft) P...",Report_CEN22FA298_105410_2_3_2025 7_50_00 AM.pdf
2,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No,,"8105 hours (Total, all aircraft), 38 hours (To...",Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
3,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes,,Page 4 of 7 CHI01FA329 Aircraft and Owner/Oper...,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
4,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No,,(Estimated) Pilot Information (A2) Certificate...,Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
5,Commercial,"45,Male",Single-engine land; Single-engine,Front sea; Multi-engine land,Helicopter,3-point,Airplane,No,None,,Class 2 With waivers/limitations,"November 16, 2022",Yes,"April 19, 2023","10570 hours (Total, all aircraft), 398 hours (...",Report_ANC23LA032_139291_2_3_2025 7_42_32 AM.pdf


In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            if "Certificate:" in page_text:
                text.append(page_text)
    return "\n".join(text)

# Function to extract pilot info
def extract_pilot_info(text, file_name):
    pattern = r"""
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Last\sFlight\sReview\sor\sEquivalent:\s*(.*?)\s*
        Flight\sTime:\s*(.*?)\s*
    """

    match = re.search(pattern, text, re.S | re.VERBOSE)
    if match:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
                "Last Flight Review or Equivalent"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]

        pilot_info = dict(zip(keys, values))


        # Extract Flight Time separately
        flight_time_match = re.search(r"Flight Time:\s*(.*)", text, re.S | re.MULTILINE)

        if flight_time_match:
            flight_time_value = flight_time_match.group(1).strip()
            pilot_info["Flight Time"] = flight_time_value if flight_time_value else "Not Available"
        else:
            pilot_info["Flight Time"] = "Not Available"

        # Move "Last Flight Review or Equivalent" after "Flight Time"
        last_flight_review = pilot_info.pop("Last Flight Review or Equivalent", "Not Found")
        pilot_info["Flight Time"] = pilot_info.pop("Flight Time", "Not Found")
        pilot_info["Last Flight Review or Equivalent"] = last_flight_review

        # Add File Name **after Last Flight Review or Equivalent**
        pilot_info["File Name"] = file_name

        return pilot_info

    return None

# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.append(res)

# Convert to DataFrame
pilot_df1 = pd.DataFrame(pilot_data)

# Ensure column order: Flight Time → Last Flight Review or Equivalent → File Name
column_order = [col for col in pilot_df1.columns if col not in ["Last Flight Review or Equivalent","Flight Time", "File Name"]] \
               + ["Last Flight Review or Equivalent","Flight Time", "File Name"]
pilot_df1 = pilot_df1[column_order]

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot2_Information.csv"
pilot_df1.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot2_Information.csv


In [ ]:
pilot_df1

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Last Flight Review or Equivalent,Flight Time,File Name
0,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No,"October 20, 2022","(Estimated) 639.49 hours (Total, all aircraft)...",Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
1,Private,"54,Male",Single-engine land,Right,None,Lap only,None,Yes,None,Yes,Class 2 With waivers/limitations,"January 19, 2022",No,,"(Estimated) 1310 hours (Total, all aircraft)\n...",Report_CEN22FA298_105410_2_3_2025 7_50_00 AM.pdf
2,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No,,"8105 hours (Total, all aircraft), 38 hours (To...",Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
3,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes,,Page 4 of 7 CHI01FA329\nAircraft and Owner/Ope...,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
4,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No,,(Estimated)\nPilot Information (A2)\nCertifica...,Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
5,Commercial,"45,Male",Single-engine land; Single-engine,Front sea; Multi-engine land,Helicopter,3-point,Airplane,No,None,,Class 2 With waivers/limitations,"November 16, 2022",Yes,"April 19, 2023","10570 hours (Total, all aircraft), 398 hours (...",Report_ANC23LA032_139291_2_3_2025 7_42_32 AM.pdf


In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            if "Certificate:" in page_text:
                text.append(page_text)
    return "\n".join(text)

# Function to extract pilot info
def extract_pilot_info(text, file_name):
    pattern = r"""
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Last\sFlight\sReview\sor\sEquivalent:\s*(.*?)\s*
    """

    match = re.search(pattern, text, re.S | re.VERBOSE)
    if match:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
                "Last Flight Review or Equivalent"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]
        pilot_info = dict(zip(keys, values))

        # Extract Flight Time separately (only up to first newline)
        flight_time_match = re.search(r"Flight Time:\s*(.*)", text, re.S | re.MULTILINE)
        if flight_time_match:
            flight_time_value = flight_time_match.group(1).strip().split("\n")[0]  # Extract only up to first newline
            pilot_info["Flight Time"] = flight_time_value if flight_time_value else "Not Available"
        else:
            pilot_info["Flight Time"] = "Not Available"

        # Move "Last Flight Review or Equivalent" after "Flight Time"
        last_flight_review = pilot_info.pop("Last Flight Review or Equivalent", "Not Available")
        pilot_info["Flight Time"] = pilot_info.pop("Flight Time", "Not Available")
        pilot_info["Last Flight Review or Equivalent"] = last_flight_review

        # Add File Name **after Last Flight Review or Equivalent**
        pilot_info["File Name"] = file_name

        return pilot_info

    return None

# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.append(res)

# Convert to DataFrame
pilot_df = pd.DataFrame(pilot_data)

# Ensure column order: Flight Time → Last Flight Review or Equivalent → File Name
column_order = [col for col in pilot_df.columns if col not in ["Last Flight Review or Equivalent", "Flight Time", "File Name"]] \
               + ["Flight Time", "Last Flight Review or Equivalent", "File Name"]
pilot_df = pilot_df[column_order]

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot3_Information.csv"
pilot_df.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot3_Information.csv


In [ ]:
pilot_df

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Flight Time,Last Flight Review or Equivalent,File Name
0,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No,"(Estimated) 639.49 hours (Total, all aircraft)...",,Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
1,Private,"54,Male",Single-engine land,Right,None,Lap only,None,Yes,None,Yes,Class 2 With waivers/limitations,"January 19, 2022",No,"(Estimated) 1310 hours (Total, all aircraft)",,Report_CEN22FA298_105410_2_3_2025 7_50_00 AM.pdf
2,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No,"8105 hours (Total, all aircraft), 38 hours (To...",,Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
3,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes,Page 4 of 7 CHI01FA329,,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
4,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No,(Estimated),,Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
5,Commercial,"45,Male",Single-engine land; Single-engine,Front sea; Multi-engine land,Helicopter,3-point,Airplane,No,None,,Class 2 With waivers/limitations,"November 16, 2022",Yes,"10570 hours (Total, all aircraft), 398 hours (...",,Report_ANC23LA032_139291_2_3_2025 7_42_32 AM.pdf


In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            if "Pilot Information" in page_text:
                text.append(page_text.split("Pilot-rated passenger Information")[0])  # Stop at the next table
    return "\n".join(text)

# Function to extract pilot info
def extract_pilot_info(text, file_name):
    pattern = r"""
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Last\sFlight\sReview\sor\sEquivalent:\s*(.*?)\s*
    """

    match = re.search(pattern, text, re.S | re.VERBOSE)
    if match:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
                "Last Flight Review or Equivalent"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]
        pilot_info = dict(zip(keys, values))

        # Extract Flight Time separately (only up to first newline)
        flight_time_match = re.search(r"Flight Time:\s*(.*)", text, re.S | re.MULTILINE)
        if flight_time_match:
            flight_time_value = flight_time_match.group(1).strip().split("\n")[0]  # Extract only up to first newline
            pilot_info["Flight Time"] = flight_time_value if flight_time_value else "Not Available"
        else:
            pilot_info["Flight Time"] = "Not Available"

        # Move "Last Flight Review or Equivalent" after "Flight Time"
        last_flight_review = pilot_info.pop("Last Flight Review or Equivalent", "Not Available")
        pilot_info["Flight Time"] = pilot_info.pop("Flight Time", "Not Available")
        pilot_info["Last Flight Review or Equivalent"] = last_flight_review

        # Add File Name **after Last Flight Review or Equivalent**
        pilot_info["File Name"] = file_name

        return pilot_info

    return None

# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.append(res)

# Convert to DataFrame
pilot_df = pd.DataFrame(pilot_data)

# Ensure column order: Flight Time → Last Flight Review or Equivalent → File Name
column_order = [col for col in pilot_df.columns if col not in ["Last Flight Review or Equivalent", "Flight Time", "File Name"]] \
               + ["Flight Time", "Last Flight Review or Equivalent", "File Name"]
pilot_df = pilot_df[column_order]

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot3_Information.csv"
pilot_df.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot3_Information.csv


In [ ]:
pilot_df

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Flight Time,Last Flight Review or Equivalent,File Name
0,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No,"(Estimated) 639.49 hours (Total, all aircraft)...",,Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
1,Private,"54,Male",Single-engine land,Right,None,Lap only,None,Yes,None,Yes,Class 2 With waivers/limitations,"January 19, 2022",No,"(Estimated) 1310 hours (Total, all aircraft)",,Report_CEN22FA298_105410_2_3_2025 7_50_00 AM.pdf
2,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes,Page 4 of 7 CHI01FA329,,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
3,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No,(Estimated),,Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
4,Commercial,"45,Male",Single-engine land; Single-engine,Front sea; Multi-engine land,Helicopter,3-point,Airplane,No,None,,Class 2 With waivers/limitations,"November 16, 2022",Yes,"10570 hours (Total, all aircraft), 398 hours (...",,Report_ANC23LA032_139291_2_3_2025 7_42_32 AM.pdf


In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            if "Certificate:" in page_text:
                text.append(page_text)
    return "\n".join(text)

# Function to extract pilot info
def extract_pilot_info(text, file_name):
    pattern = r"""
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Last\sFlight\sReview\sor\sEquivalent:\s*(.*?)\s*
        Flight\sTime:\s*(.*?)(?:\n|$)
    """

    match = re.search(pattern, text, re.S | re.VERBOSE)
    if match:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
                "Last Flight Review or Equivalent","Flight Time"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]

        pilot_info = dict(zip(keys, values))



        return pilot_info

    return None

# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.append(res)

# Convert to DataFrame
pilot_df1 = pd.DataFrame(pilot_data)





In [ ]:
pilot_df1

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Last Flight Review or Equivalent,Flight Time
0,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No,"October 20, 2022","(Estimated) 639.49 hours (Total, all aircraft)..."
1,Private,"54,Male",Single-engine land,Right,None,Lap only,None,Yes,None,Yes,Class 2 With waivers/limitations,"January 19, 2022",No,,"(Estimated) 1310 hours (Total, all aircraft)"
2,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No,,Page 5 of 8 CEN23LA211
3,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes,,Page 4 of 7 CHI01FA329
4,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No,,(Estimated)
5,Commercial,"45,Male",Single-engine land; Single-engine,Front sea; Multi-engine land,Helicopter,3-point,Airplane,No,None,,Class 2 With waivers/limitations,"November 16, 2022",Yes,"April 19, 2023","10570 hours (Total, all aircraft), 398 hours (..."


In [ ]:
# prompt: pilot_df1 save to csv

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot_Inf.csv"
pilot_df1.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot_Inf.csv


In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            if "Certificate:" in page_text:
                text.append(page_text)
    return "\n".join(text)

# Function to extract pilot info
def extract_pilot_info(text, file_name):
    pattern = r"""
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Flight\sTime:\s*(.*?)(?=\n\S|Page)
    """

    match = re.search(pattern, text, re.S | re.VERBOSE)
    if match:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
                "Flight Time"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]

        pilot_info = dict(zip(keys, values))

        # Add File Name **after Flight Time**
        pilot_info["File Name"] = file_name

        return pilot_info

    return None

# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.append(res)

# Convert to DataFrame
pilot_df1 = pd.DataFrame(pilot_data)

# Ensure column order: Flight Time → File Name
column_order = [col for col in pilot_df1.columns if col not in ["Flight Time", "File Name"]] + ["Flight Time", "File Name"]
pilot_df1 = pilot_df1[column_order]

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot2_Information.csv"
pilot_df1.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot2_Information.csv


In [ ]:
pilot_df1

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Flight Time,File Name
0,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No Last Flight Review or Equivalent: October 2...,"(Estimated) 639.49 hours (Total, all aircraft)...",Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
1,Private,"54,Male",Single-engine land,Right,None,Lap only,None,Yes,None,Yes,Class 2 With waivers/limitations,"January 19, 2022",No Last Flight Review or Equivalent:,"(Estimated) 1310 hours (Total, all aircraft)",Report_CEN22FA298_105410_2_3_2025 7_50_00 AM.pdf
2,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No Last Flight Review or Equivalent:,,Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
3,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes Last Flight Review or Equivalent:,,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
4,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No Last Flight Review or Equivalent:,(Estimated),Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
5,Commercial,"45,Male",Single-engine land; Single-engine,Front sea; Multi-engine land,Helicopter,3-point,Airplane,No,None,,Class 2 With waivers/limitations,"November 16, 2022",Yes Last Flight Review or Equivalent: April 19...,"10570 hours (Total, all aircraft), 398 hours (...",Report_ANC23LA032_139291_2_3_2025 7_42_32 AM.pdf


In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            if "Certificate:" in page_text:
                text.append(page_text)
    return "\n".join(text)

# Function to extract pilot info
def extract_pilot_info(text, file_name):
    pattern = r"""
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*
        Flight\sTime:\s*((?:.*?\n)*?)(?=\n\S|Page)
    """

    match = re.search(pattern, text, re.S | re.VERBOSE)
    if match:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
                "Flight Time"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]

        pilot_info = dict(zip(keys, values))

        # Add File Name **after Flight Time**
        pilot_info["File Name"] = file_name

        return pilot_info

    return None

# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.append(res)

# Convert to DataFrame
pilot_df1 = pd.DataFrame(pilot_data)

# Ensure column order: Flight Time → File Name
column_order = [col for col in pilot_df1.columns if col not in ["Flight Time", "File Name"]] + ["Flight Time", "File Name"]
pilot_df1 = pilot_df1[column_order]

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot2_Information.csv"
pilot_df1.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot2_Information.csv


Two tables in single pdf

In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            text.append(page_text)
    return "\n".join(text)

# Function to extract pilot info
def extract_pilot_info(text, file_name):
    pattern = r"""
        Pilot\sInformation.*?\n
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*
        Flight\sTime:\s*((?:.*?\n)*?)(?=\n\S|Page|Pilot\sInformation)
    """

    matches = re.finditer(pattern, text, re.S | re.VERBOSE)
    pilot_records = []

    for match in matches:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
                "Flight Time"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]

        pilot_info = dict(zip(keys, values))

        # Add File Name **after Flight Time**
        pilot_info["File Name"] = file_name
        pilot_records.append(pilot_info)

    return pilot_records

# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.extend(res)  # Append multiple records

# Convert to DataFrame
pilot_df1 = pd.DataFrame(pilot_data)

# Ensure column order: Flight Time → File Name
column_order = [col for col in pilot_df1.columns if col not in ["Flight Time", "File Name"]] + ["Flight Time", "File Name"]
pilot_df1 = pilot_df1[column_order]

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot2_Information.csv"
pilot_df1.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot2_Information.csv


In [ ]:
pilot_df1

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Flight Time,File Name
0,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes Last Flight Review or Equivalent:,,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
1,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No Last Flight Review or Equivalent:,(Estimated),Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
2,Private,"68,Male",Single-engine land,Front,None,4-point,None,No,None,,Class 3 With waivers/limitations,"October 7, 2021","No Last Flight Review or Equivalent: April 4, ...","4950 hours (Total, all aircraft), 227 hours (T...",Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
3,Commercial,"45,Male",Single-engine land; Single-engine,Front sea; Multi-engine land,Helicopter,3-point,Airplane,No,None,,Class 2 With waivers/limitations,"November 16, 2022",Yes Last Flight Review or Equivalent: April 19...,"10570 hours (Total, all aircraft), 398 hours (...",Report_ANC23LA032_139291_2_3_2025 7_42_32 AM.pdf
4,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No Last Flight Review or Equivalent:,,Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
5,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No Last Flight Review or Equivalent: October 2...,"(Estimated) 639.49 hours (Total, all aircraft)...",Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
6,Private,"54,Male",Single-engine land,Right,None,Lap only,None,Yes,None,Yes,Class 2 With waivers/limitations,"January 19, 2022",No Last Flight Review or Equivalent:,"(Estimated) 1310 hours (Total, all aircraft) P...",Report_CEN22FA298_105410_2_3_2025 7_50_00 AM.pdf


In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            text.append(page_text)
    return "\n".join(text)

# Function to extract pilot info
def extract_pilot_info(text, file_name):
    pattern = r"""
        Pilot\sInformation.*?\n
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*
        Flight\sTime:\s*((?:.*?\n)*?)(?=\n\S|Page|Aircraft\sand\sOwner/Operator\sInformation|Pilot-rated\spassenger\sInformation|Passenger\sInformation)
    """

    matches = re.finditer(pattern, text, re.S | re.VERBOSE)
    pilot_records = []

    for match in matches:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
                "Flight Time"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]

        pilot_info = dict(zip(keys, values))

        # Add File Name **after Flight Time**
        pilot_info["File Name"] = file_name
        pilot_records.append(pilot_info)

    return pilot_records

# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.extend(res)  # Append multiple records

# Convert to DataFrame
pilot_df1 = pd.DataFrame(pilot_data)

# Ensure column order: Flight Time → File Name
column_order = [col for col in pilot_df1.columns if col not in ["Flight Time", "File Name"]] + ["Flight Time", "File Name"]
pilot_df1 = pilot_df1[column_order]

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot2_Information.csv"
pilot_df1.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot2_Information.csv


In [ ]:
pilot_df1

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Flight Time,File Name
0,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes Last Flight Review or Equivalent:,,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
1,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No Last Flight Review or Equivalent:,(Estimated) Pilot Information (A2) Certificate...,Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
2,Commercial,"45,Male",Single-engine land; Single-engine,Front sea; Multi-engine land,Helicopter,3-point,Airplane,No,None,,Class 2 With waivers/limitations,"November 16, 2022",Yes Last Flight Review or Equivalent: April 19...,"10570 hours (Total, all aircraft), 398 hours (...",Report_ANC23LA032_139291_2_3_2025 7_42_32 AM.pdf
3,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No Last Flight Review or Equivalent:,,Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
4,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No Last Flight Review or Equivalent: October 2...,"(Estimated) 639.49 hours (Total, all aircraft)...",Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
5,Private,"54,Male",Single-engine land,Right,None,Lap only,None,Yes,None,Yes,Class 2 With waivers/limitations,"January 19, 2022",No Last Flight Review or Equivalent:,"(Estimated) 1310 hours (Total, all aircraft)",Report_CEN22FA298_105410_2_3_2025 7_50_00 AM.pdf


In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            text.append(page_text)
    return "\n".join(text)

# Function to extract pilot info
def extract_pilot_info(text, file_name):
    pattern = r"""
        Pilot\sInformation.*?\n
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*
        Flight\sTime:\s*((?:.*?\n)*?)(?=\n\S|Page|Aircraft\sand\sOwner/Operator\sInformation|Pilot-rated\spassenger\sInformation|Passenger\sInformation)
    """

    matches = re.finditer(pattern, text, re.S | re.VERBOSE)
    pilot_records = []

    for match in matches:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
                "Flight Time"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]

        pilot_info = dict(zip(keys, values))

        # Add File Name **after Flight Time**
        pilot_info["File Name"] = file_name
        pilot_records.append(pilot_info)

    return pilot_records

# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.extend(res)  # Append multiple records

# Convert to DataFrame
pilot_df1 = pd.DataFrame(pilot_data)

# Ensure column order: Flight Time → File Name
column_order = [col for col in pilot_df1.columns if col not in ["Flight Time", "File Name"]] + ["Flight Time", "File Name"]
pilot_df1 = pilot_df1[column_order]

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot2_Information.csv"
pilot_df1.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot2_Information.csv


### Data Extraction

This is good but inconsistency in columns

In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            text.append(page_text)
    return "\n".join(text)

# Function to extract pilot info
def extract_pilot_info(text, file_name):
    pattern = r"""
        Pilot\sInformation.*?\n
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Last\sFlight\sReview\sor\sEquivalent:\s*(.*?)\s*
        Flight\sTime:\s*((?:.*?\n)*?)(?=\n\S|Co-pilot\sInformation|Flight\sinstructor\sInformation|Pilot|Page|Aircraft\sand\sOwner/Operator\sInformation|Pilot-rated\spassenger\sInformation|Passenger\sInformation|Pilot-rated\spassenger\sInformation)
    """

    matches = re.finditer(pattern, text, re.S | re.VERBOSE)
    pilot_records = []

    for match in matches:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot","Last Flight Review or Equivalent",
                "Flight Time"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]

        pilot_info = dict(zip(keys, values))

        # Add File Name **after Flight Time**
        pilot_info["File Name"] = file_name
        pilot_records.append(pilot_info)

    return pilot_records

# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.extend(res)  # Append multiple records

# Convert to DataFrame
pilot_df1 = pd.DataFrame(pilot_data)

# Ensure column order: Flight Time → File Name
column_order = [col for col in pilot_df1.columns if col not in ["Flight Time", "File Name"]] + ["Flight Time", "File Name"]
pilot_df1 = pilot_df1[column_order]

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot2_Information.csv"
pilot_df1.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot2_Information.csv


In [ ]:
# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot2_Information.csv"
pilot_df1.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")

✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot2_Information.csv


In [ ]:
pilot_df1

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Last Flight Review or Equivalent,Flight Time,File Name
0,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes,,,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
1,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No,,(Estimated),Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
2,Private,"68,Male",Single-engine land,Front,None,4-point,None,No,None,,Class 3 With waivers/limitations,"October 7, 2021",No,"April 4, 2021","4950 hours (Total, all aircraft), 227 hours (T...",Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
3,Commercial,"45,Male",Single-engine land; Single-engine,Front sea; Multi-engine land,Helicopter,3-point,Airplane,No,None,,Class 2 With waivers/limitations,"November 16, 2022",Yes,"April 19, 2023","10570 hours (Total, all aircraft), 398 hours (...",Report_ANC23LA032_139291_2_3_2025 7_42_32 AM.pdf
4,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No,,,Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
5,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No,"October 20, 2022","(Estimated) 639.49 hours (Total, all aircraft)...",Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
6,Private,"54,Male",Single-engine land,Right,None,Lap only,None,Yes,None,Yes,Class 2 With waivers/limitations,"January 19, 2022",No,,"(Estimated) 1310 hours (Total, all aircraft)",Report_CEN22FA298_105410_2_3_2025 7_50_00 AM.pdf


In [ ]:

import pandas as pd

# Load the CSV file into a pandas DataFrame.
pilot_df = pd.read_csv("/content/drive/MyDrive/Pilot2_Information.csv")

# Print the DataFrame.
pilot_df


,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Last Flight Review or Equivalent,Flight Time,File Name
0,Private,"74,Male",Single-engine land,Left,NaN,Lap only,NaN,Yes,NaN,NaN,Class 3 With waivers/limitations,"April 24, 2023",No,"April 7, 2024","(Estimated) 202.5 hours (Total, all aircraft),...",Report_CEN24LA237_194531_2_5_2025 1_17_38 AM.pdf
1,Private,"60,Male",Single-engine land,Front,NaN,3-point,NaN,No,NaN,Yes,Class 3 With waivers/limitations,"September 8, 2022",No,"October 20, 2022","(Estimated) 639.49 hours (Total, all aircraft)...",Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
2,Private,"60,Male",Single-engine land,Left,NaN,Lap only,NaN,No,NaN,NaN,Class 2 With waivers/limitations,"January 17, 2023",No,"March 4, 2023","343 hours (Total, all aircraft), 55 hours (Tot...",Report_CEN23LA238_192359_2_3_2025 7_58_20 AM.pdf
3,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,NaN,NaN,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes,NaN,NaN,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
4,Airline transport; Commercial,"50,Male",Single-engine land; Multi-engine,Left land,Balloon,NaN,Airplane,Yes,Airplane multi-engine; Airplane,single-engine; Instrument airplane,Class 1 With waivers/limitations,"February 15, 2023",Yes,NaN,"15552 hours (Total, all aircraft), 752 hours (...",Report_CEN23LA286_192566_2_3_2025 7_46_13 AM.pdf
5,Private,"56,Male",Single-engine land,Left,NaN,NaN,Airplane,NaN,NaN,NaN,Class 3,"March 4, 2022",No,NaN,(Estimated),Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
6,Private,"68,Male",Single-engine land,Front,NaN,4-point,NaN,No,NaN,NaN,Class 3 With waivers/limitations,"October 7, 2021",No,"April 4, 2021","4950 hours (Total, all aircraft), 227 hours (T...",Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
7,Private,"37,Male",Single-engine land; Single-engine,Unknown sea,NaN,Unknown,NaN,NaN,NaN,NaN,Class 3 Without,"November 1, 2018 waivers/limitations",No,NaN,"(Estimated) 246 hours (Total, all aircraft), 2...",Report_ANC22LA059_105533_2_5_2025 1_16_22 AM.pdf
8,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,NaN,4-point,NaN,No,NaN,NaN,Class 2 With waivers/limitations,"July 29, 2020",No,NaN,NaN,Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
9,Commercial; Flight instructor,"62,Male",Single-engine land; Multi-engine,Rear land,NaN,4-point,Airplane,NaN,Airplane single-engine,NaN,Class 2 With waivers/limitations,"April 19, 2022",Yes,"October 13, 2020","5904 hours (Total, all aircraft), 3730 hours (...",Report_ERA22LA379_105776_2_5_2025 1_18_07 AM.pdf


In [ ]:
# prompt: pilot_df from this dataframe get only 8th row

# Assuming pilot_df is already loaded as shown in your provided code.

eighth_row = pilot_df.iloc[8]  # 8th row (index 7)
eighth_row


,8
Certificate,"Airline transport; Commercial; Age: 53,Male Fl..."
Age,"64,Female"
Airplane Rating(s),Single-engine land
Seat Occupied,Left
Other Aircraft Rating(s),NaN
Restraint Used,4-point
Instrument Rating(s),NaN
Second Pilot Present,No
Instructor Rating(s),NaN
Toxicology Performed,NaN


In [ ]:
# prompt: eighth_row = pilot_df.iloc[8]  # 8th row (index 7)
# eighth_row
# now for eigth row what is in Certificate column i want to see its value

print(eighth_row['Certificate'])


Airline transport; Commercial; Age: 53,Male Flight instructor Airplane Rating(s): Single-engine land; Multi-engine Seat Occupied: Right land Other Aircraft Rating(s): Glider Restraint Used: 4-point Instrument Rating(s): Airplane Second Pilot Present: No Instructor Rating(s): Airplane multi-engine; Airplane Toxicology Performed: single-engine; Instrument airplane Medical Certification: BasicMed With waivers/limitations Last FAA Medical Exam: August 25, 2022 Occupational Pilot: No Last Flight Review or Equivalent: September 8, 2022 Flight Time: 8105 hours (Total, all aircraft), 38 hours (Total, this make and model), 7281 hours (Pilot In Command, all aircraft), 53 hours (Last 90 days, all aircraft), 12 hours (Last 30 days, all aircraft), 0 hours (Last 24 hours, all aircraft) Pilot-rated passenger Information Certificate: Commercial


In [ ]:
import easyocr  # Uses GPU for OCR
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor  # Multiprocessing for CPU-heavy tasks

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU for OCR

# Extract text from PDFs, using OCR if necessary
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:  # If text extraction fails, use OCR
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            text.append(page_text)
    return "\n".join(text)

# Function to extract pilot info
def extract_pilot_info(text, file_name):
    if "Pilot Information" not in text or "Certificate:" not in text:
        return []  # Skip if required sections don't exist

    pattern = r"""
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Last\sFlight\sReview\sor\sEquivalent:\s*(.*?)\s*
        Flight\sTime:\s*((?:.*?\n)*?)(?=\n\S|Co-pilot\sInformation|Flight\sinstructor\sInformation|Pilot|Page|Aircraft\sand\sOwner/Operator\sInformation|Pilot-rated\spassenger\sInformation|Passenger\sInformation|Pilot-rated\spassenger\sInformation)
    """

    matches = re.finditer(pattern, text, re.S | re.VERBOSE)
    pilot_records = []

    for match in matches:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot", "Last Flight Review or Equivalent",
                "Flight Time"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]

        pilot_info = dict(zip(keys, values))

        # Add File Name **after Flight Time**
        pilot_info["File Name"] = file_name
        pilot_records.append(pilot_info)

    return pilot_records

# Process all PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:  # Use multiple CPU cores
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.extend(res)  # Append multiple records

# Convert to DataFrame
pilot_df2 = pd.DataFrame(pilot_data)

# Ensure column order: Flight Time → File Name
column_order = [col for col in pilot_df2.columns if col not in ["Flight Time", "File Name"]] + ["Flight Time", "File Name"]
pilot_df2 = pilot_df2[column_order]

# Save to CSV
output_csv = r"/content/drive/MyDrive/Pilot2_Information.csv"
pilot_df2.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot2_Information.csv


In [ ]:
pilot_df2

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Last Flight Review or Equivalent,Flight Time,File Name
0,Private,"74,Male",Single-engine land,Left,None,Lap only,None,Yes,None,,Class 3 With waivers/limitations,"April 24, 2023",No,"April 7, 2024","(Estimated) 202.5 hours (Total, all aircraft),...",Report_CEN24LA237_194531_2_5_2025 1_17_38 AM.pdf
1,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No,"October 20, 2022","(Estimated) 639.49 hours (Total, all aircraft)...",Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
2,,"64,Male",,Rear,,3-point,,No,,,,,No,,,Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
3,Private,"60,Male",Single-engine land,Left,None,Lap only,None,No,None,,Class 2 With waivers/limitations,"January 17, 2023",No,"March 4, 2023","343 hours (Total, all aircraft), 55 hours (Tot...",Report_CEN23LA238_192359_2_3_2025 7_58_20 AM.pdf
4,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes,,,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
5,Airline transport; Commercial,"50,Male",Single-engine land; Multi-engine,Left land,Balloon,,Airplane,Yes,Airplane multi-engine; Airplane,single-engine; Instrument airplane,Class 1 With waivers/limitations,"February 15, 2023",Yes,,"15552 hours (Total, all aircraft), 752 hours (...",Report_CEN23LA286_192566_2_3_2025 7_46_13 AM.pdf
6,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No,,(Estimated),Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
7,Private,"68,Male",Single-engine land,Front,None,4-point,None,No,None,,Class 3 With waivers/limitations,"October 7, 2021",No,"April 4, 2021","4950 hours (Total, all aircraft), 227 hours (T...",Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
8,Private,"37,Male",Single-engine land; Single-engine,Unknown sea,None,Unknown,None,,None,,Class 3 Without,"November 1, 2018 waivers/limitations",No,,"(Estimated) 246 hours (Total, all aircraft), 2...",Report_ANC22LA059_105533_2_5_2025 1_16_22 AM.pdf
9,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No,,,Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf


In [ ]:
import easyocr
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)

# Extract text from PDFs using OCR if needed
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            text.append(page_text)
    return "\n".join(text)

# Extract pilot information using a more robust pattern
def extract_pilot_info(text, file_name):
    if "Pilot Information" not in text or "Certificate:" not in text:
        return []

    # Improved regex pattern for better handling of fields
    pattern = r"""
        Pilot\sInformation.*?
        Certificate:\s*(.*?)\s+Age:\s*(\d+,\w+)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Last\sFlight\sReview\sor\sEquivalent:\s*(.*?)\s*
        Flight\sTime:\s*((?:.*?\n)*?)(?=\n\S|Co-pilot\sInformation|Page|$)
    """

    matches = re.finditer(pattern, text, re.S | re.VERBOSE)
    pilot_records = []

    for match in matches:
        keys = ["Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
                "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
                "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
                "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
                "Last Flight Review or Equivalent", "Flight Time"]

        values = [match.group(i+1).replace("\n", " ").strip() if match.group(i+1) else "" for i in range(len(keys))]

        pilot_info = dict(zip(keys, values))
        pilot_info["File Name"] = file_name  # Track the source file
        pilot_records.append(pilot_info)

    return pilot_records

# Process PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

# Run parallel extraction
pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.extend(res)

# Convert extracted data to DataFrame
pilot_df4 = pd.DataFrame(pilot_data)

# Ensure proper column order
column_order = [col for col in pilot_df4.columns if col not in ["Flight Time", "File Name"]] + ["Flight Time", "File Name"]
pilot_df4 = pilot_df4[column_order]

# Save results to CSV
output_csv = r"/content/drive/MyDrive/Pilot4_Information.csv"
pilot_df4.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot4_Information.csv


In [ ]:
pilot_df4

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Last Flight Review or Equivalent,Flight Time,File Name
0,Private,"74,Male",Single-engine land,Left,None,Lap only,None,Yes,None,,Class 3 With waivers/limitations,"April 24, 2023",No,"April 7, 2024","(Estimated) 202.5 hours (Total, all aircraft),...",Report_CEN24LA237_194531_2_5_2025 1_17_38 AM.pdf
1,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No,"October 20, 2022","(Estimated) 639.49 hours (Total, all aircraft)...",Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
2,Private,"60,Male",Single-engine land,Left,None,Lap only,None,No,None,,Class 2 With waivers/limitations,"January 17, 2023",No,"March 4, 2023","343 hours (Total, all aircraft), 55 hours (Tot...",Report_CEN23LA238_192359_2_3_2025 7_58_20 AM.pdf
3,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes,,,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
4,Airline transport; Commercial,"50,Male",Single-engine land; Multi-engine,Left land,Balloon,,Airplane,Yes,Airplane multi-engine; Airplane,single-engine; Instrument airplane,Class 1 With waivers/limitations,"February 15, 2023",Yes,,"15552 hours (Total, all aircraft), 752 hours (...",Report_CEN23LA286_192566_2_3_2025 7_46_13 AM.pdf
5,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No,,(Estimated) Pilot Information (A2) Certificate...,Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
6,Private,"37,Male",Single-engine land; Single-engine,Unknown sea,None,Unknown,None,,None,,Class 3 Without,"November 1, 2018 waivers/limitations",No,,"(Estimated) 246 hours (Total, all aircraft), 2...",Report_ANC22LA059_105533_2_5_2025 1_16_22 AM.pdf
7,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No,,,Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
8,Commercial; Flight instructor,"62,Male",Single-engine land; Multi-engine,Rear land,None,4-point,Airplane,,Airplane single-engine,,Class 2 With waivers/limitations,"April 19, 2022",Yes,"October 13, 2020","5904 hours (Total, all aircraft), 3730 hours (...",Report_ERA22LA379_105776_2_5_2025 1_18_07 AM.pdf
9,Commercial,"60,Male",Single-engine land; Multi-engine,Left land,None,3-point,Airplane,No,None,,Class 3 With waivers/limitations,"May 20, 2021",No,"June 6, 2022","905 hours (Total, all aircraft), 332 hours (To...",Report_CEN23LA014_106135_2_5_2025 1_16_56 AM.pdf


In [ ]:
import easyocr
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)

# Extract text from PDFs using OCR if needed
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            text.append(page_text)
    return "\n".join(text)

# Extract only Pilot Information
def extract_pilot_info(text, file_name):
    if "Pilot Information" not in text or "Certificate:" not in text:
        return []

    pattern = r"""
        Pilot\sInformation.*?
        Certificate:\s*(.*?)\s+Age:\s*(\d+\,?\w*)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Last\sFlight\sReview\sor\sEquivalent:\s*(.*?)\s*
        Flight\sTime:\s*((?:.*?\n)*?)(?=\n\S|Page|Aircraft|$)
    """

    matches = re.finditer(pattern, text, re.S | re.VERBOSE)
    pilot_records = []

    for match in matches:
        keys = [
            "Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
            "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
            "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
            "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
            "Last Flight Review or Equivalent", "Flight Time"
        ]

        values = [
            match.group(i + 1).replace("\n", " ").strip() if match.group(i + 1) else ""
            for i in range(len(keys))
        ]
        pilot_info = dict(zip(keys, values))
        pilot_info["File Name"] = file_name
        pilot_records.append(pilot_info)

    return pilot_records

# Process PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

# Run parallel extraction
pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.extend(res)

# Convert extracted data to DataFrame
pilot_df1 = pd.DataFrame(pilot_data)

# Ensure proper column order
column_order = [col for col in pilot_df1.columns if col not in ["Flight Time", "File Name"]] + ["Flight Time", "File Name"]
pilot_df1 = pilot_df1[column_order]

# Save results to CSV
output_csv = r"/content/drive/MyDrive/Pilot2_Information.csv"
pilot_df1.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")


✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot2_Information.csv


In [ ]:
pilot_df1

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Last Flight Review or Equivalent,Flight Time,File Name
0,Private,"74,Male",Single-engine land,Left,None,Lap only,None,Yes,None,,Class 3 With waivers/limitations,"April 24, 2023",No,"April 7, 2024","(Estimated) 202.5 hours (Total, all aircraft),...",Report_CEN24LA237_194531_2_5_2025 1_17_38 AM.pdf
1,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No,"October 20, 2022","(Estimated) 639.49 hours (Total, all aircraft)...",Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
2,Private,"60,Male",Single-engine land,Left,None,Lap only,None,No,None,,Class 2 With waivers/limitations,"January 17, 2023",No,"March 4, 2023","343 hours (Total, all aircraft), 55 hours (Tot...",Report_CEN23LA238_192359_2_3_2025 7_58_20 AM.pdf
3,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes,,,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
4,Airline transport; Commercial,"50,Male",Single-engine land; Multi-engine,Left land,Balloon,,Airplane,Yes,Airplane multi-engine; Airplane,single-engine; Instrument airplane,Class 1 With waivers/limitations,"February 15, 2023",Yes,,"15552 hours (Total, all aircraft), 752 hours (...",Report_CEN23LA286_192566_2_3_2025 7_46_13 AM.pdf
5,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No,,(Estimated) Pilot Information (A2) Certificate...,Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
6,Private,"37,Male",Single-engine land; Single-engine,Unknown sea,None,Unknown,None,,None,,Class 3 Without,"November 1, 2018 waivers/limitations",No,,"(Estimated) 246 hours (Total, all aircraft), 2...",Report_ANC22LA059_105533_2_5_2025 1_16_22 AM.pdf
7,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No,,,Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
8,Commercial; Flight instructor,"62,Male",Single-engine land; Multi-engine,Rear land,None,4-point,Airplane,,Airplane single-engine,,Class 2 With waivers/limitations,"April 19, 2022",Yes,"October 13, 2020","5904 hours (Total, all aircraft), 3730 hours (...",Report_ERA22LA379_105776_2_5_2025 1_18_07 AM.pdf
9,Commercial,"60,Male",Single-engine land; Multi-engine,Left land,None,3-point,Airplane,No,None,,Class 3 With waivers/limitations,"May 20, 2021",No,"June 6, 2022","905 hours (Total, all aircraft), 332 hours (To...",Report_CEN23LA014_106135_2_5_2025 1_16_56 AM.pdf


In [ ]:
import easyocr
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)

# Extract text from PDFs using OCR if needed
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            text.append(page_text)
    return "\n".join(text)

# Extract only Pilot Information, stopping before co-pilot information
def extract_pilot_info(text, file_name):
    if "Pilot Information" not in text or "Certificate:" not in text:
        return []

    pattern = r"""
        Pilot\sInformation.*?
        Certificate:\s*(.*?)\s+Age:\s*(\d+\,?\w*)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Last\sFlight\sReview\sor\sEquivalent:\s*(.*?)\s*
        Flight\sTime:\s*((?:.*?\n)*?)(?=\n\S|Pilot-rated\spassenger\sInformation|Page|Aircraft|$)
    """

    matches = re.finditer(pattern, text, re.S | re.VERBOSE)
    pilot_records = []

    for match in matches:
        keys = [
            "Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
            "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
            "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
            "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
            "Last Flight Review or Equivalent", "Flight Time"
        ]

        values = [
            match.group(i + 1).replace("\n", " ").strip() if match.group(i + 1) else ""
            for i in range(len(keys))
        ]
        pilot_info = dict(zip(keys, values))
        pilot_info["File Name"] = file_name
        pilot_records.append(pilot_info)

    return pilot_records

# Process PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

# Run parallel extraction
pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.extend(res)

# Convert extracted data to DataFrame
pilot_df1 = pd.DataFrame(pilot_data)

# Ensure proper column order
column_order = [col for col in pilot_df1.columns if col not in ["Flight Time", "File Name"]] + ["Flight Time", "File Name"]
pilot_df1 = pilot_df1[column_order]

# Save results to CSV
output_csv = r"/content/drive/MyDrive/Pilot2_Information.csv"
pilot_df1.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete✅ Extracted pilot information saved to /content/drive/MyDrive/Pilot2_Information.csv


In [ ]:
pilot_df1

,Certificate,Age,Airplane Rating(s),Seat Occupied,Other Aircraft Rating(s),Restraint Used,Instrument Rating(s),Second Pilot Present,Instructor Rating(s),Toxicology Performed,Medical Certification,Last FAA Medical Exam,Occupational Pilot,Last Flight Review or Equivalent,Flight Time,File Name
0,Private,"74,Male",Single-engine land,Left,None,Lap only,None,Yes,None,,Class 3 With waivers/limitations,"April 24, 2023",No,"April 7, 2024","(Estimated) 202.5 hours (Total, all aircraft),...",Report_CEN24LA237_194531_2_5_2025 1_17_38 AM.pdf
1,Private,"60,Male",Single-engine land,Front,None,3-point,None,No,None,Yes,Class 3 With waivers/limitations,"September 8, 2022",No,"October 20, 2022","(Estimated) 639.49 hours (Total, all aircraft)...",Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
2,Private,"60,Male",Single-engine land,Left,None,Lap only,None,No,None,,Class 2 With waivers/limitations,"January 17, 2023",No,"March 4, 2023","343 hours (Total, all aircraft), 55 hours (Tot...",Report_CEN23LA238_192359_2_3_2025 7_58_20 AM.pdf
3,Commercial; Flight instructor,"41,Male",Single-engine land; Multi-engine,Left land,None,,Airplane,Yes,Airplane single-engine; Instrument,Yes airplane,Class 1 None,"May 28, 2001",Yes,,,Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
4,Airline transport; Commercial,"50,Male",Single-engine land; Multi-engine,Left land,Balloon,,Airplane,Yes,Airplane multi-engine; Airplane,single-engine; Instrument airplane,Class 1 With waivers/limitations,"February 15, 2023",Yes,,"15552 hours (Total, all aircraft), 752 hours (...",Report_CEN23LA286_192566_2_3_2025 7_46_13 AM.pdf
5,Private,"56,Male",Single-engine land,Left,None,,Airplane,,None,,Class 3,"March 4, 2022",No,,(Estimated) Pilot Information (A2) Certificate...,Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
6,Private,"37,Male",Single-engine land; Single-engine,Unknown sea,None,Unknown,None,,None,,Class 3 Without,"November 1, 2018 waivers/limitations",No,,"(Estimated) 246 hours (Total, all aircraft), 2...",Report_ANC22LA059_105533_2_5_2025 1_16_22 AM.pdf
7,"Airline transport; Commercial; Age: 53,Male Fl...","64,Female",Single-engine land,Left,None,4-point,None,No,None,,Class 2 With waivers/limitations,"July 29, 2020",No,,,Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
8,Commercial; Flight instructor,"62,Male",Single-engine land; Multi-engine,Rear land,None,4-point,Airplane,,Airplane single-engine,,Class 2 With waivers/limitations,"April 19, 2022",Yes,"October 13, 2020","5904 hours (Total, all aircraft), 3730 hours (...",Report_ERA22LA379_105776_2_5_2025 1_18_07 AM.pdf
9,Commercial,"60,Male",Single-engine land; Multi-engine,Left land,None,3-point,Airplane,No,None,,Class 3 With waivers/limitations,"May 20, 2021",No,"June 6, 2022","905 hours (Total, all aircraft), 332 hours (To...",Report_CEN23LA014_106135_2_5_2025 1_16_56 AM.pdf


In [ ]:
import easyocr
import pdfplumber
import pandas as pd
import os
import re
from concurrent.futures import ProcessPoolExecutor

pdf_dir = r"/content/drive/MyDrive/CP"
reader = easyocr.Reader(['en'], gpu=True)

# Extract text from PDFs using OCR if needed
def extract_text_from_pdf(pdf_path):
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if not page_text:
                page_img = page.to_image()
                ocr_text = reader.readtext(page_img.original, detail=0)
                page_text = " ".join(ocr_text)
            text.append(page_text)
    return "\n".join(text)

# Extract only Pilot Information section
def extract_pilot_info(text, file_name):
    if "Pilot Information" not in text:
        return []

    # Isolate the Pilot Information section by splitting text
    pilot_info_section = re.search(r"Pilot Information(.*?)\n(?:Pilot-rated passenger Information|Aircraft and Owner/Operator Information|Page|$)", text, re.S)
    if not pilot_info_section:
        return []

    pilot_text = pilot_info_section.group(1)

    pattern = r"""
        Certificate:\s*(.*?)\s+Age:\s*(\d+\,?\w*)\s*
        Airplane\sRating\(s\):\s*(.*?)\s*Seat\sOccupied:\s*(.*?)\s*
        Other\sAircraft\sRating\(s\):\s*(.*?)\s*Restraint\sUsed:\s*(.*?)\s*
        Instrument\sRating\(s\):\s*(.*?)\s*Second\sPilot\sPresent:\s*(.*?)\s*
        Instructor\sRating\(s\):\s*(.*?)\s*Toxicology\sPerformed:\s*(.*?)\s*
        Medical\sCertification:\s*(.*?)\s*Last\sFAA\sMedical\sExam:\s*(.*?)\s*
        Occupational\sPilot:\s*(.*?)\s*Last\sFlight\sReview\sor\sEquivalent:\s*(.*?)\s*
        Flight\sTime:\s*((?:.*?\n)*?)$
    """

    match = re.search(pattern, pilot_text, re.S | re.VERBOSE)
    if not match:
        return []

    keys = [
        "Certificate", "Age", "Airplane Rating(s)", "Seat Occupied",
        "Other Aircraft Rating(s)", "Restraint Used", "Instrument Rating(s)",
        "Second Pilot Present", "Instructor Rating(s)", "Toxicology Performed",
        "Medical Certification", "Last FAA Medical Exam", "Occupational Pilot",
        "Last Flight Review or Equivalent", "Flight Time"
    ]

    values = [
        match.group(i + 1).replace("\n", " ").strip() if match.group(i + 1) else ""
        for i in range(len(keys))
    ]
    pilot_info = dict(zip(keys, values))
    pilot_info["File Name"] = file_name

    return [pilot_info]

# Process PDFs using multiprocessing
def process_pdf(file):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, file)
        pdf_text = extract_text_from_pdf(pdf_path)
        return extract_pilot_info(pdf_text, file)

# Run parallel extraction
pilot_data = []
with ProcessPoolExecutor(max_workers=4) as executor:
    results = executor.map(process_pdf, os.listdir(pdf_dir))
    for res in results:
        if res:
            pilot_data.extend(res)

# Convert extracted data to DataFrame
pilot_df1 = pd.DataFrame(pilot_data)

# Ensure proper column order
column_order = [col for col in pilot_df1.columns if col not in ["Flight Time", "File Name"]] + ["Flight Time", "File Name"]
pilot_df1 = pilot_df1[column_order]

# Save results to CSV
output_csv = r"/content/drive/MyDrive/Pilot2_Information.csv"
pilot_df1.to_csv(output_csv, index=False)
print(f"✅ Extracted pilot information saved to {output_csv}")
